In [1]:
# --- 必要なパッケージの読み込み ---
library(caret)
library(Metrics)
library(ggplot2)
library(lattice)
library(dplyr) # %>% を利用 (OOD分割関数で使用)
library(kknn)  # kknnメソッドの実行に必要

# --- OOD分割関数 ---
create_ood_split <- function(df, feature_cols, ood_ratio = 0.1) {
    df_features <- df[, feature_cols, drop = FALSE]
    dist_matrix <- dist(df_features, method = "euclidean")
    
    # %>% は dplyr または magrittr 由来
    avg_distances <- as.matrix(dist_matrix) %>% colMeans() 

    num_ood <- floor(nrow(df) * ood_ratio)
    if (num_ood == 0 && nrow(df) > 0) num_ood <- 1 

    ood_indices <- order(avg_distances, decreasing = TRUE)[1:num_ood]

    test_set <- df[ood_indices, ]
    train_set <- df[-ood_indices, ]

    cat(paste0("  OOD Split: Total=", nrow(df), ", Train=", nrow(train_set), ", Test(OOD)=", nrow(test_set), "\n"))

    return(list(train = train_set, test = test_set, ood_indices = ood_indices))
}

# --- 対象ファイル名一覧 ---
file_names <- c(
    "n_base.csv", "n_base_OH.csv", "n_base_FP.csv",
    "n_all.csv", "n_all_OH.csv", "n_all_FP.csv",
    "m_base.csv", "m_base_OH.csv", "m_base_FP.csv",
    "m_all.csv", "m_all_OH.csv", "m_all_FP.csv"
)

# データパスと日付
base_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/"
today <- format(Sys.Date(), "%Y%m%d")

# 評価指標の初期化 (kをkmaxに、OODメトリックを追加)
target_vars <- c("Jsc", "Voc", "FF", "PCEmax")
metric_names <- c("Best_kmax", "R2", "RMSE", "MAE", "RPD", "n_samples", "n_features", "OOD_R2", "OOD_RMSE")
result_matrix <- matrix(nrow = length(metric_names) * length(target_vars), ncol = length(file_names))
rownames(result_matrix) <- as.vector(t(outer(metric_names, target_vars, paste, sep = "_")))
colnames(result_matrix) <- file_names

# --- メイン処理ループ ---
for (file in file_names) {
    filepath <- paste0(base_path, file)
    cat("\n=== Processing:", file, "===\n")
    df_all <- read.csv(filepath) 
    
    feature_vars <- setdiff(colnames(df_all), target_vars)

    for (target_var in target_vars) {
        cat("\n---\nTraining model for:", target_var, "on", file, "\n")
        
        # 目的変数に欠損値がない完全な行のみを抽出
        df_temp <- df_all[, c(feature_vars, target_var)]
        df_complete <- df_temp[complete.cases(df_temp), ]

        if (nrow(df_complete) < 20) {
            cat("  Skipping due to insufficient complete data (N < 20).\n")
            next
        }

        # === 1. OOD分割の実行 (FPデータのみ) ===
        is_fp_data <- grepl("_FP\\.csv$", file) 
        
        if (is_fp_data) {
            cat("  Performing OOD split based on structural distance...\n")
            split_list <- create_ood_split(df_complete, feature_vars, ood_ratio = 0.1) 
            df_train_cv <- split_list$train # 学習・CV用データ
            df_ood_test <- split_list$test  # OODテスト用データ
        } else {
            df_train_cv <- df_complete # FPデータ以外は全て学習・CV用
            df_ood_test <- NULL
        }
        
        # === 2. kknnモデルの訓練 (訓練データでCV) ===
        # 学習と交差検証の設定
        tune_grid <- expand.grid(kmax = seq(1, 20, by = 2)) # kをkmaxに変更
        ctrl <- trainControl(method = "cv", number = 10, savePredictions = "final")

        model <- train(
            formula(paste(target_var, "~ .")),
            data = df_train_cv,
            method = "kknn", # knnをkknnに変更
            metric = "RMSE",
            trControl = ctrl,
            tuneGrid = tune_grid,
            preProcess = c("center", "scale")
        )

        # === 3. CV結果 (内挿性能) の計算と格納 ===
        pred_df <- model$pred
        # 最良kmaxのみ抽出
        pred_df <- pred_df[pred_df$kmax == model$bestTune$kmax, ]
        
        if (nrow(pred_df) > 0) {
            pred <- pred_df$pred
            obs <- pred_df$obs
        
            R2 <- round(cor(pred, obs)^2, 3)
            RMSE_val <- round(rmse(obs, pred), 3)
            MAE_val <- round(mae(obs, pred), 3)
            RPD_val <- round(sd(obs) / RMSE_val, 3)
        } else {
            warning("No predictions matched bestTune. Skipping this model evaluation.")
            R2 <- RMSE_val <- MAE_val <- RPD_val <- NA
        }

        best_kmax <- model$bestTune$kmax
        cat("Best kmax value:", best_kmax, "\n")
        cat(file, target_var, ": CV R2 =", R2, ", RMSE =", RMSE_val, "\n")

        # 結果保存 (CV性能)
        result_matrix[paste0("Best_kmax_", target_var), file] <- best_kmax # kをkmaxに変更
        result_matrix[paste0("R2_", target_var), file] <- R2
        result_matrix[paste0("RMSE_", target_var), file] <- RMSE_val
        result_matrix[paste0("MAE_", target_var), file] <- MAE_val
        result_matrix[paste0("RPD_", target_var), file] <- RPD_val
        result_matrix[paste0("n_samples_", target_var), file] <- nrow(df_complete)
        result_matrix[paste0("n_features_", target_var), file] <- length(feature_vars)


        # === 4. OODテスト性能 (外挿性能) の計算と格納 ===
        if (is_fp_data && !is.null(df_ood_test) && nrow(df_ood_test) > 0) {
            cat("  Evaluating OOD test set performance...\n")
            
            # 予測
            ood_preds <- predict(model, newdata = df_ood_test)
            ood_obs <- df_ood_test[[target_var]]
            
            # OODメトリック計算
            ood_R2 <- round(cor(ood_obs, ood_preds)^2, 3)
            ood_RMSE_val <- round(rmse(ood_obs, ood_preds), 3)
            
            # OOD性能格納
            result_matrix[paste0("OOD_R2_", target_var), file] <- ood_R2
            result_matrix[paste0("OOD_RMSE_", target_var), file] <- ood_RMSE_val
            
            cat(paste0("  OOD R2 = ", ood_R2, ", OOD RMSE = ", ood_RMSE_val, "\n"))
        } else {
            # FPデータ以外はNA
            result_matrix[paste0("OOD_R2_", target_var), file] <- NA
            result_matrix[paste0("OOD_RMSE_", target_var), file] <- NA
        }

        # === 5. モデルとプロットの保存 ===
        
        # モデル保存
        model_data_bundle <- list(model = model, training_data = df_train_cv, ood_test_data = df_ood_test)
        rds_file <- paste0("20250702_model_data_", tools::file_path_sans_ext(file), "_", target_var, "_kknn_", today, ".rds")
        saveRDS(model_data_bundle, file = rds_file)

        # プロット保存 (PDFは省略し、データ保存に集中)
    }
}

# 結果の保存
output_file <- paste0("20250702_kknn_summary_", today, ".csv")
write.csv(result_matrix, output_file, row.names = TRUE)
cat("\nSummary saved as", output_file, "\n")

Loading required package: ggplot2

Loading required package: lattice


Attaching package: 'Metrics'


The following objects are masked from 'package:caret':

    precision, recall



Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union




ERROR: Error in library(kknn): there is no package called 'kknn'
